## Запуск анализа данных в неинтерактивном режиме

In [5]:
%cd /content/drive/MyDrive/big_data

/content/drive/MyDrive/big_data


In [6]:
!spark-submit --master local[*] /content/drive/MyDrive/big_data/L1-Introduction_to_Apache_Spark/L1_noninteractive_bike_analysis_python.py

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/27 19:43:39 INFO SparkContext: Running Spark version 4.0.2
26/04/27 19:43:39 INFO SparkContext: OS info Linux, 6.6.113+, amd64
26/04/27 19:43:39 INFO SparkContext: Java version 17.0.18
26/04/27 19:43:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/27 19:43:39 INFO ResourceUtils: ==============================================================
26/04/27 19:43:39 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/27 19:43:39 INFO ResourceUtils: ==============================================================
26/04/27 19:43:39 INFO SparkContext: Submitted application: Lab1_Script
26/04/27 19:43:39 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name:

## Решение задач

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("BikeAnalysis_Tasks").master("local[*]").getOrCreate()

trips_df = spark.read.csv("/content/drive/MyDrive/big_data/data/trips.csv", header=True, inferSchema=True)
stations_df = spark.read.csv("/content/drive/MyDrive/big_data/data/stations.csv", header=True, inferSchema=True)

trips_df.createOrReplaceTempView("trips")
stations_df.createOrReplaceTempView("stations")

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Найти велосипед с максимальным временем пробега.

In [8]:
max_running_time_bike = spark.sql("""
SELECT bike_id, SUM(duration) as total_duration
FROM trips
GROUP BY bike_id
ORDER BY total_duration DESC
LIMIT 1
""")

max_running_time_bike.show()

+-------+--------------+
|bike_id|total_duration|
+-------+--------------+
|    535|      18611693|
+-------+--------------+



## Найти наибольшее геодезическое расстояние между станциями.

In [9]:
max_geo_length = spark.sql("""
SELECT
    s1.name AS station_A,
    s2.name AS station_B,
    (6371 * ACOS(
        COS(RADIANS(s1.lat)) * COS(RADIANS(s2.lat)) * COS(RADIANS(s2.long) - RADIANS(s1.long)) +
        SIN(RADIANS(s1.lat)) * SIN(RADIANS(s2.lat))
    )) AS distance_km
FROM stations s1 CROSS JOIN stations s2
WHERE s1.id < s2.id
ORDER BY distance_km DESC
LIMIT 1
""")
max_geo_length.show()

+--------------------+--------------------+-----------------+
|           station_A|           station_B|      distance_km|
+--------------------+--------------------+-----------------+
|SJSU - San Salvad...|Embarcadero at Sa...|69.92087595421542|
+--------------------+--------------------+-----------------+



## Найти путь велосипеда с максимальным временем пробега через станции.

In [10]:
bike_path = spark.sql("""
SELECT start_date, start_station_name, end_station_name
FROM trips
WHERE bike_id = (
    SELECT bike_id
    FROM trips
    GROUP BY bike_id
    ORDER BY SUM(duration) DESC
    LIMIT 1
)
ORDER BY TO_TIMESTAMP(start_date, 'M/d/yyyy H:mm')
""")
bike_path.show(50, truncate=False)

+---------------+---------------------------------------------+----------------------------------------+
|start_date     |start_station_name                           |end_station_name                        |
+---------------+---------------------------------------------+----------------------------------------+
|8/29/2013 19:32|Post at Kearney                              |San Francisco Caltrain (Townsend at 4th)|
|8/29/2013 21:38|San Francisco Caltrain (Townsend at 4th)     |San Francisco Caltrain 2 (330 Townsend) |
|8/30/2013 8:40 |San Francisco Caltrain 2 (330 Townsend)      |Market at Sansome                       |
|8/30/2013 9:10 |Market at Sansome                            |2nd at South Park                       |
|9/1/2013 12:58 |2nd at Townsend                              |Davis at Jackson                        |
|9/5/2013 11:59 |San Francisco City Hall                      |Civic Center BART (7th at Market)       |
|9/6/2013 10:55 |Civic Center BART (7th at Market)     

## Найти количество велосипедов в системе.

In [11]:
total_bikes = spark.sql("""
SELECT COUNT(DISTINCT bike_id) AS total_bikes
FROM trips
""")
total_bikes.show()

+-----------+
|total_bikes|
+-----------+
|        700|
+-----------+



## Найти пользователей потративших на поездки более 3 часов.

In [12]:
more_3h_users = spark.sql("""
SELECT
    id AS trip_id,
    duration,
    start_date,
    subscription_type,
    zip_code
FROM trips
WHERE duration > 10800
ORDER BY duration DESC
""")
more_3h_users.show(50, truncate=False)
# в датасете нет id пользователей, поэтому выводится информация которая известна о пользователях через поездки

+-------+--------+----------------+-----------------+--------+
|trip_id|duration|start_date      |subscription_type|zip_code|
+-------+--------+----------------+-----------------+--------+
|568474 |17270400|12/6/2014 21:59 |Customer         |95531   |
|825850 |2137000 |6/28/2015 21:50 |Customer         |97213   |
|750192 |1852590 |5/2/2015 6:17   |Subscriber       |94024   |
|841176 |1133540 |7/10/2015 10:35 |Customer         |94306   |
|111309 |722236  |11/30/2013 13:29|Customer         |94301   |
|522337 |720454  |10/30/2014 8:29 |Customer         |94010   |
|323594 |716480  |6/13/2014 16:57 |Subscriber       |94131   |
|361321 |715339  |7/13/2014 5:50  |Customer         |nil     |
|774999 |688899  |5/20/2015 15:27 |Customer         |nil     |
|635260 |655939  |2/8/2015 3:05   |Customer         |89451   |
|237942 |644771  |4/6/2014 3:37   |Customer         |94014   |
|129504 |619322  |12/18/2013 9:16 |Subscriber       |94041   |
|745640 |611240  |4/29/2015 9:41  |Customer         |81